# Task 1: Customer Churn Classification

**Goal:** Build and evaluate supervised classification models to predict whether a telecom customer will churn.

**Requirements covered:**
- Data preprocessing, train/test split, cross-validation
- Compare Logistic Regression and Random Forest
- Report Accuracy, Precision, Recall, F1, and ROC-AUC

**Dataset:** IBM Telco Customer Churn (public sample)

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42
print("Libraries loaded successfully.")

## 2. Load dataset

The notebook downloads the Telco Customer Churn CSV if it is not already present under `data/`.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "Telco-Customer-Churn.csv"
DATA_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

if not DATA_PATH.exists():
    print("Downloading Telco Customer Churn dataset...")
    df = pd.read_csv(DATA_URL)
    df.to_csv(DATA_PATH, index=False)
else:
    df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
df.head()

## 3. Exploratory data analysis

In [ ]:
print("Column types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))
print("\nTarget distribution:")
print(df["Churn"].value_counts(normalize=True).round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.countplot(data=df, x="Churn", ax=axes[0, 0], palette="Set2")
axes[0, 0].set_title("Churn class balance")

sns.histplot(data=df, x="tenure", hue="Churn", bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Tenure by churn")

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1, 0], palette="Set2")
axes[1, 0].set_title("Monthly charges by churn")

contract_churn = (
    df.groupby(["Contract", "Churn"]).size().reset_index(name="count")
)
sns.barplot(data=contract_churn, x="Contract", y="count", hue="Churn", ax=axes[1, 1])
axes[1, 1].set_title("Contract type vs churn")
axes[1, 1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 4. Data preprocessing

- Drop customer ID (non-predictive identifier)
- Convert `TotalCharges` to numeric (some blank strings exist)
- Encode the target (`Yes` → 1, `No` → 0)
- Build a preprocessing pipeline: impute + scale numeric features; impute + one-hot encode categorical features

In [ ]:
data = df.copy()
data = data.drop(columns=["customerID"])
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data["Churn"] = data["Churn"].map({"Yes": 1, "No": 0})

print("Missing TotalCharges after conversion:", data["TotalCharges"].isna().sum())
print("Target counts:\n", data["Churn"].value_counts())

X = data.drop(columns=["Churn"])
y = data["Churn"]

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

## 5. Train / test split

Stratified 80/20 split to preserve churn class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print("Train churn rate:", round(y_train.mean(), 3))
print("Test churn rate:", round(y_test.mean(), 3))

## 6. Model training & cross-validation

Compare **Logistic Regression** and **Random Forest** with 5-fold stratified cross-validation on the training set.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
pipelines = {}

for name, estimator in models.items():
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )
    pipelines[name] = pipe

    cv_results = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    row = {"Model": name}
    for metric in scoring:
        scores = cv_results[f"test_{metric}"]
        row[metric] = f"{scores.mean():.3f} ± {scores.std():.3f}"
        row[f"{metric}_mean"] = scores.mean()
    cv_rows.append(row)
    print(f"Finished CV for {name}")

cv_df = pd.DataFrame(cv_rows)
display_cols = ["Model", "accuracy", "precision", "recall", "f1", "roc_auc"]
print("\n5-Fold Cross-Validation Results (mean ± std):")
cv_df[display_cols]

## 7. Final evaluation on held-out test set

In [ ]:
def evaluate_model(name, pipe, X_tr, y_tr, X_te, y_te):
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred),
        "Recall": recall_score(y_te, y_pred),
        "F1": f1_score(y_te, y_pred),
        "ROC-AUC": roc_auc_score(y_te, y_proba),
    }

    print(f"\n=== {name} — Classification Report ===")
    print(classification_report(y_te, y_pred, target_names=["No Churn", "Churn"]))
    return metrics, y_pred, y_proba, pipe


test_rows = []
fitted = {}
predictions = {}

for name, pipe in pipelines.items():
    metrics, y_pred, y_proba, fitted_pipe = evaluate_model(
        name, pipe, X_train, y_train, X_test, y_test
    )
    test_rows.append(metrics)
    fitted[name] = fitted_pipe
    predictions[name] = {"y_pred": y_pred, "y_proba": y_proba}

test_df = pd.DataFrame(test_rows).set_index("Model")
print("\nHeld-out Test Metrics:")
test_df.round(3)

## 8. Plots: confusion matrices & ROC curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, name in zip(axes, predictions):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions[name]["y_pred"],
        display_labels=["No Churn", "Churn"],
        cmap="Blues",
        ax=ax,
        colorbar=False,
    )
    ax.set_title(f"Confusion Matrix — {name}")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name in predictions:
    RocCurveDisplay.from_predictions(
        y_test,
        predictions[name]["y_proba"],
        name=name,
        ax=ax,
    )

ax.plot([0, 1], [0, 1], "k--", label="Chance")
ax.set_title("ROC Curves — Test Set")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = test_df.reset_index().melt(
    id_vars="Model", var_name="Metric", value_name="Score"
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=plot_df, x="Metric", y="Score", hue="Model", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Test-set metric comparison")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=8)
plt.tight_layout()
plt.show()

## 9. Feature importance (Random Forest)

Useful for model interpretability / reporting.

In [ ]:
rf_pipe = fitted["Random Forest"]
ohe = rf_pipe.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
cat_names = ohe.get_feature_names_out(categorical_features)
feature_names = np.concatenate([numeric_features, cat_names])

importances = rf_pipe.named_steps["model"].feature_importances_
imp_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=imp_df, x="importance", y="feature", ax=ax, palette="viridis")
ax.set_title("Top 15 Random Forest feature importances")
plt.tight_layout()
plt.show()

imp_df

## 10. Model selection summary

Select the better model primarily by **F1** and **ROC-AUC** (important for imbalanced churn), then confirm with recall for catching churners.

In [ ]:
best_model = test_df["F1"].idxmax()
print("Selected model (highest F1 on test set):", best_model)
print("\nSelected model metrics:")
print(test_df.loc[best_model].round(3))

print("\nCross-validation F1 means:")
print(
    cv_df[["Model", "f1_mean", "roc_auc_mean"]]
    .sort_values("f1_mean", ascending=False)
    .round(3)
    .to_string(index=False)
)

---

### Responsible AI notes
- **Class imbalance:** ~27% churn; used stratified splits and `class_weight="balanced"`.
- **Fairness:** features such as gender/senior citizen should be audited for disparate impact before production use.
- **Privacy:** customer IDs were dropped; avoid storing raw PII in model artifacts.
- **Interpretability:** Random Forest importances and Logistic Regression coefficients support stakeholder reporting.